<a href="https://colab.research.google.com/github/springboardmentorLive/Reality_AI/blob/ananyarohidasshet/IS_Movie_Recommendation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:

import pandas as pd
import numpy as np
from zipfile import ZipFile
import tensorflow as tf
from tensorflow import keras
from pathlib import Path
import matplotlib.pyplot as plt

In [ ]:
from zipfile import ZipFile
from pathlib import Path
import pandas as pd

# Path where your zip file exists
movielens_zipped_file = "/content/ml-latest-small.zip"   # Colab path
# movielens_zipped_file = "/mnt/data/ml-latest-small.zip"  # if using provided file

keras_datasets_path = Path(movielens_zipped_file).parent
movielens_dir = keras_datasets_path / "ml-latest-small"

# Extract only if not already extracted
if not movielens_dir.exists():
    with ZipFile(movielens_zipped_file, "r") as zip:
        print("Extracting files...")
        zip.extractall(path=keras_datasets_path)
        print("Done!")

# Load CSV files
ratings_file = movielens_dir / "ratings.csv"
tags_file = movielens_dir / "tags.csv"
movies_file = movielens_dir / "movies.csv"

df = pd.read_csv(ratings_file)
tags = pd.read_csv(tags_file)
movies = pd.read_csv(movies_file)

print(df.head())
print(movies.head())


Extracting files...
Done!
   userId  movieId  rating  timestamp
0       1        1     4.0  964982703
1       1        2     5.0  964982224
2       2        2     3.5  964981247
3       2        3     4.5  964982931
4       3        1     5.0  964982400
   movieId                               title  \
0        1                    Toy Story (1995)   
1        2                      Jumanji (1995)   
2        3             Grumpier Old Men (1995)   
3        4            Waiting to Exhale (1995)   
4        5  Father of the Bride Part II (1995)   

                                        genres  
0  Adventure|Animation|Children|Comedy|Fantasy  
1                   Adventure|Children|Fantasy  
2                               Comedy|Romance  
3                         Comedy|Drama|Romance  
4                                       Comedy  


In [ ]:
df.head

<bound method NDFrame.head of    userId  movieId  rating  timestamp
0       1        1     4.0  964982703
1       1        2     5.0  964982224
2       2        2     3.5  964981247
3       2        3     4.5  964982931
4       3        1     5.0  964982400>

In [ ]:
df.describe()

,userId,movieId,rating,timestamp
count,5.00000,5.00000,5.00000,5.000000e+00
mean,1.80000,1.80000,4.40000,9.649823e+08
std,0.83666,0.83666,0.65192,6.490666e+02
min,1.00000,1.00000,3.50000,9.649812e+08
25%,1.00000,1.00000,4.00000,9.649822e+08
50%,2.00000,2.00000,4.50000,9.649824e+08
75%,2.00000,2.00000,5.00000,9.649827e+08
max,3.00000,3.00000,5.00000,9.649829e+08


In [ ]:

df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5 entries, 0 to 4
Data columns (total 4 columns):
 #   Column     Non-Null Count  Dtype  
---  ------     --------------  -----  
 0   userId     5 non-null      int64  
 1   movieId    5 non-null      int64  
 2   rating     5 non-null      float64
 3   timestamp  5 non-null      int64  
dtypes: float64(1), int64(3)
memory usage: 292.0 bytes


In [ ]:

# Map user ID to a "user vector" via an embedding matrix
user_ids = df["userId"].unique().tolist()
user2user_encoded = {x: i for i, x in enumerate(user_ids)}
userencoded2user = {i: x for i, x in enumerate(user_ids)}

# Map movies ID to a "movies vector" via an embedding matrix
movie_ids = df["movieId"].unique().tolist()
movie2movie_encoded = {x: i for i, x in enumerate(movie_ids)}
movie_encoded2movie = {i: x for i, x in enumerate(movie_ids)}

df["user"] = df["userId"].map(user2user_encoded)
df["movie"] = df["movieId"].map(movie2movie_encoded)

num_users = len(user2user_encoded)
num_movies = len(movie_encoded2movie)
df['rating'] = df['rating'].values.astype(np.float32)

# min and max ratings will be used to normalize the ratings later
min_rating = min(df["rating"])
max_rating = max(df["rating"])

print(f"Number of users: {num_users}, Number of Movies: {num_movies}, Min Rating: {min_rating}, Max Rating: {max_rating}")
df = df.sample(frac=1, random_state=42)
x = df[["user", "movie"]].values

# Normalizing the targets between 0 and 1. Makes it easy to train.
y = df["rating"].apply(lambda x: (x - min_rating) / (max_rating - min_rating)).values

# Assuming training on 90% of the data and validating on 100%
train_indices = int(0.9 * df.shape[0])
x_train, x_val, y_train, y_val = (
    x[:train_indices],
    x[train_indices:],
    y[:train_indices],
    y[train_indices:],
)

Number of users: 3, Number of Movies: 3, Min Rating: 3.5, Max Rating: 5.0


In [ ]:
user_id = 1

print("Showing recommendations for user: {}".format(user_id))
print("====" * 9)

print("Movies with high ratings from user")
print("----" * 8)

movies_watched_by_user = df[df.userId == user_id]

top_movies_user = (
    movies_watched_by_user.sort_values(by="rating", ascending=False)
    .head(5)
    .movieId.values
)

movie_df_rows = movies[movies["movieId"].isin(top_movies_user)]
for row in movie_df_rows.itertuples():
    print(row.title, ":", row.genres)

print("----" * 8)
print("Top 10 movie recommendations")
print("----" * 8)

recommended_movie_ids = (
    df.groupby("movieId")["rating"]
    .mean()
    .sort_values(ascending=False)
    .head(10)
    .index
)

recommended_movies = movies[movies["movieId"].isin(recommended_movie_ids)]
for row in recommended_movies.itertuples():
    print(row.title, ":", row.genres)


Showing recommendations for user: 1
Movies with high ratings from user
--------------------------------
Toy Story (1995) : Adventure|Animation|Children|Comedy|Fantasy
Jumanji (1995) : Adventure|Children|Fantasy
--------------------------------
Top 10 movie recommendations
--------------------------------
Toy Story (1995) : Adventure|Animation|Children|Comedy|Fantasy
Jumanji (1995) : Adventure|Children|Fantasy
Grumpier Old Men (1995) : Comedy|Romance
